# Fine-tuning RML — Qwen2.5-7B-Instruct
## Progetto: Voci dal Fronte

Fine-tuning con LoRA + SFT su **81,080 esempi ChatML** estratti da Supabase.

**Requisiti GPU:** T4 16GB (free Colab) o superiore

### Dataset Breakdown:
- `aya_ita_chatml` (40) — Italian NLU
- `commandnet_chatml` (10,000) — Command understanding
- `muninn_ww1_chatml` (28,700) — WW1 documents
- `quandho_chatml` (1,800) — Italian QA
- `train_merged` (40,540) — Merged specializzato eventi bellici

In [ ]:
# Installa dipendenze
!pip install -q transformers peft trl bitsandbytes datasets accelerate httpx python-dotenv
!pip install -q flash-attn --no-build-isolation

In [ ]:
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
print(f'CUDA: {torch.version.cuda}')

In [ ]:
# Configurazione Supabase (inserisci le tue chiavi)
import os
os.environ['SUPABASE_URL'] = ''  # <-- inserisci URL Supabase
os.environ['SUPABASE_SERVICE_ROLE_KEY'] = ''  # <-- inserisci Service Role Key

# Oppure carica da Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/drive/MyDrive/voci_dal_fronte/.env .

In [ ]:
import httpx
import json

SUPABASE_URL = os.environ['SUPABASE_URL']
SUPABASE_KEY = os.environ['SUPABASE_SERVICE_ROLE_KEY']

def fetch_training_data():
    headers = {
        'apikey': SUPABASE_KEY,
        'Authorization': f'Bearer {SUPABASE_KEY}',
    }
    all_records = []
    offset = 0
    batch_size = 1000
    while True:
        r = httpx.get(
            f'{SUPABASE_URL}/rest/v1/ml_training_chatml',
            headers=headers,
            params={'select': 'dataset_name,messages', 'order': 'id', 'offset': str(offset), 'limit': str(batch_size)},
            timeout=30,
        )
        if r.status_code != 200:
            print(f'Error: {r.status_code}')
            break
        batch = r.json()
        if not batch:
            break
        for rec in batch:
            msgs = rec['messages']
            if isinstance(msgs, str):
                msgs = json.loads(msgs)
            all_records.append({'messages': msgs})
        offset += len(batch)
        if offset % 10000 == 0:
            print(f'  {offset:,} record...')
    print(f'Totale: {len(all_records):,} record')
    return all_records

records = fetch_training_data()

In [ ]:
from datasets import Dataset

SYSTEM_PROMPT = (
    'Sei un ricercatore storico specializzato negli eventi bellici del Novecento, '
    'con focus su Prima e Seconda Guerra Mondiale, Internati Militari Italiani (IMI), '
    'caduti, decorati e fonti archivistiche italiane. '
    'Rispondi in italiano con accuratezza storica, citando fonti quando possibile.'
)

processed = []
for rec in records:
    messages = rec['messages']
    if not messages or messages[0].get('role') != 'system':
        messages = [{'role': 'system', 'content': SYSTEM_PROMPT}] + messages
    else:
        messages[0]['content'] = SYSTEM_PROMPT
    messages = [m for m in messages if m.get('content', '').strip()]
    if len(messages) >= 2:
        processed.append({'messages': messages})

ds = Dataset.from_list(processed)
split = ds.train_test_split(test_size=0.05, seed=42)
print(f'Train: {len(split["train"]):,}, Val: {len(split["test"]):,}')

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, padding_side='right')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    attn_implementation='flash_attention_2',
)
model = prepare_model_for_kbit_training(model)

# LoRA
lora_config = LoraConfig(
    r=64,
    lora_alpha=128,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir='./qwen_voci_dal_fronte',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    logging_steps=10,
    save_strategy='steps',
    save_steps=500,
    eval_strategy='steps',
    eval_steps=500,
    max_seq_length=2048,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    report_to='none',
    seed=42,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    processing_class=tokenizer,
)

print('Avvio training...')
train_result = trainer.train()
print(f'Loss finale: {train_result.metrics["train_loss"]:.4f}')

In [ ]:
# Salva modello
trainer.save_model('./qwen_voci_dal_fronte/final')
tokenizer.save_pretrained('./qwen_voci_dal_fronte/final')
print('Modello salvato!')

# Upload su Google Drive
from google.colab import drive
drive.mount('/content/drive')
!cp -r ./qwen_voci_dal_fronte/final /content/drive/MyDrive/voci_dal_fronte_model/
print('Modello copiato su Google Drive!')

In [ ]:
# Test inferenza
from peft import PeftModel

test_messages = [
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user', 'content': 'Cosa accadde durante la Battaglia di Caporetto nel 1917?'},
]

input_text = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(input_text, return_tensors='pt').to('cuda')

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.7, do_sample=True)

response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(response)

In [ ]:
# Merge LoRA + Export GGUF per LMStudio
!pip install -q llama-cpp-python

from peft import PeftModel
from transformers import AutoModelForCausalLM

# Merge
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map='cpu', trust_remote_code=True)
model_merged = PeftModel.from_pretrained(base_model, './qwen_voci_dal_fronte/final')
model_merged = model_merged.merge_and_unload()
model_merged.save_pretrained('./qwen_voci_merged')
tokenizer.save_pretrained('./qwen_voci_merged')

print('Modello merged! Per GGUF:')
print('  !git clone https://github.com/ggerganov/llama.cpp')
print('  !python llama.cpp/convert_hf_to_gguf.py ./qwen_voci_merged --outtype q4_k_m')